# 07 · Synapse: explaining decisions in three languages
Synapse is the read-only language layer. It never changes a light (proven by `scripts/check_architecture.py`).

* **Narrator:** turns each decision-log entry into a sentence in English, French or Arabic (templates, no LLM).
* **Explainer:** answers "why?" questions from the log, with citations, or **abstains**.
* **Optional LLM agent** (Groq or Anthropic key): searches the log with tools and writes the answer;
  its citations are checked against the log.

We use the decision log of the **combined** scenario (rain, an incident, ambulances).

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))          # the project root, so `traffix` can be imported
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
from traffix.core import config
from traffix.evaluation import notebook_tools as nt

In [ ]:
from traffix.evaluation.benchmark_synapse import fixed_log
from traffix.agents.network.explainer import explain
entries, narr = fixed_log()
print(len(entries), "decisions in the reference log")

## 1. One decision, three languages

In [ ]:
for reason in ["EMERGENCY_GRANTED_SWITCH", "INCIDENT_SUSPECTED", "WEATHER_RAIN"]:
    e = next((x for x in entries if x["reason"] == reason), None)
    if e:
        print(f"#{e['id']} {reason}")
        for lang in ("en", "fr", "ar"): print(f"   {lang}: {narr.sentence(e, lang)}")
        print()

## 2. Asking questions (template engine)
Topic questions get the rule behind the behaviour plus a summary of the run; questions about one vehicle get its story.

In [ ]:
for q, lang in [("Why did ambulance 0 get green?", "en"),
                ("Qu'est-ce que la pluie a changé ?", "fr"),
                ("لماذا حصلت الحافلات المتأخرة على الأولوية؟", "ar")]:
    a = explain(entries, q, engine="template", narrator=narr, lang=lang)
    print(f"Q ({lang}): {q}\n{a['answer'][:700]}\n   → {len(a['citations'])} citations\n")

## 3. A trick question: it must abstain, not invent

In [ ]:
a = explain(entries, "What happened with ambulance 9?", engine="template", narrator=narr, lang="en")
print(a["answer"])

## 4. The LLM agent (only with a key)

In [ ]:
from traffix.synapse import llm
if llm.available():
    a = explain(entries, "Why did ambulance 0 get green?", engine="agent", narrator=narr, lang="fr")
    print(a.get("engine"), "| tool calls:", [c["tool"] for c in a.get("tool_calls", [])])
    print(a["answer"])
else:
    print("No GROQ_API_KEY / ANTHROPIC_API_KEY set: the template engine above is what runs offline.")

## 5. Synapse benchmark (30 questions, incl. trick questions)

In [ ]:
s = nt.load_benchmark("synapse_summary.csv")
display(s) if s is not None else print("Not run yet: python scripts/benchmark.py synapse")

## 6. The architecture guarantees

In [ ]:
import subprocess
print(subprocess.run([sys.executable, os.path.join("..", "scripts", "check_architecture.py")], capture_output=True, text=True).stdout)